[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/su-ntu-ctp/6m-data-3.10-Transformers-Attention-GenAI/blob/main/notebooks/optional_extensions.ipynb)

**Where to run this notebook**

- **Locally (VS Code + Jupyter)**: just open the notebook and pick the `dsai-m3` kernel. The repo's `.env` file handles thread caps automatically.
- **Colab (recommended if you don't have a GPU)**: click the badge above, then **Runtime → Change runtime type → T4 GPU**, then run the setup cell below. It clones the repo, installs missing deps, and `cd`s into the right working directory.


In [1]:
# === Colab-compat setup (no-op when running locally) ===
# This whole IN_COLAB block is skipped when you run locally; it only fires on Colab.
import os, sys

# Detect Colab by checking whether its module was imported into the session.
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # On Colab there is no repo on disk, so we clone it and install the deps Colab lacks.
    REPO_URL = "https://github.com/su-ntu-ctp/6m-data-3.10-Transformers-Attention-GenAI.git"
    REPO_DIR = "/content/6m-data-3.10-Transformers-Attention-GenAI"
    LESSON_DIR = "notebooks"

    if not os.path.exists(REPO_DIR):
        # Clone once; -q keeps the output quiet.
        print(f"Cloning repo into {REPO_DIR} ...")
        os.system(f"git clone -q {REPO_URL} {REPO_DIR}")

    # Move into the notebooks dir so relative paths match the local layout.
    os.chdir(f"{REPO_DIR}/{LESSON_DIR}")
    print(f"Working directory: {os.getcwd()}")

    # Colab has torch + torchvision pre-installed. Install the rest.
    os.system("pip install -q sentence-transformers transformers")
    print("Colab setup done.")

# Threading caps — picked up by .env locally, set explicitly here as a fallback.
# Harmless if already set. (Loop form prevents Jupyter from auto-displaying the return value.)
# setdefault only sets the var if it is not already defined, so it never clobbers .env.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")   # silence a harmless warning
# (Old OMP/MKL one-thread caps removed — they slowed CPU-only machines.)

# L10 · Optional Extensions

> ⏱️ **~45 min** if you do all three experiments · fully optional · dip into any one on its own

> *Self-study material — three deep dives that build production GenAI skill. Skip any of them; do them in any order.*

This notebook covers:

1. **Prompt engineering** — how the prompt structure shapes LLM behaviour
2. **Sampling deep-dive** — temperature vs top-p vs top-k, with measurements
3. **When to fine-tune** — the decision tree for prompt → RAG → fine-tune

## Common setup

In [2]:
import os
# Env vars below must be set BEFORE importing torch/transformers to take effect.
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'  # allow duplicate OpenMP runtime (macOS libomp conflict)
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'  # don't phone home to Hugging Face
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'  # hide noisy info/warning logs

import warnings
warnings.filterwarnings('ignore')  # keep the notebook output clean

# Core imports: timing, the model runtime (torch), and the HF loader classes.
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# (Removed the old one-CPU-thread cap — it made CPU-only machines much slower.)
torch.manual_seed(42)     # fix the RNG so model loading/setup is repeatable

# A small instruction-tuned LLM (360M params) — fast enough to run on CPU.
MODEL = 'HuggingFaceTB/SmolLM2-360M-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL)        # text <-> token ids
llm = AutoModelForCausalLM.from_pretrained(MODEL)       # the actual generative model

def generate(messages, max_new_tokens=120, **kwargs):
    # Turn a chat-style messages list into the exact prompt string the model expects.
    # add_generation_prompt=True appends the assistant turn marker so the model continues.
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # Encode the prompt to token ids (the model's numeric input).
    input_ids = tokenizer(prompt, return_tensors='pt')['input_ids']
    # Generate continuation tokens. **kwargs lets callers pass sampling params (temperature, etc).
    out = llm.generate(input_ids, max_new_tokens=max_new_tokens,
                       pad_token_id=tokenizer.eos_token_id, **kwargs)
    # Slice off the prompt tokens so we keep only the newly generated reply, then decode to text.
    return tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

print('Model ready.')

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model ready.


---

## Experiment 1 — Prompt engineering: zero-shot vs few-shot vs role-conditioned

The SAME underlying model produces dramatically different output depending on how you frame the prompt. Let's test three patterns on a single task: classifying customer complaints into a category.

In [3]:
# Zero-shot: just describe the task -- no examples, the model must already "know" it.
zero_shot = [
    {'role': 'user', 'content': 'Classify this complaint into ONE of: shipping, quality, sizing, billing.\nComplaint: "The hem is unravelling after one wash."\nCategory:'}
]

# Few-shot: give the model worked examples first; it infers the pattern + output format from them.
few_shot = [
    {'role': 'user', 'content': '''Classify each complaint into ONE of: shipping, quality, sizing, billing.

Examples:
Complaint: "Box arrived crushed and the package was open."
Category: shipping

Complaint: "Charged twice for the same order."
Category: billing

Complaint: "Runs about 2 inches small."
Category: sizing

Complaint: "The hem is unravelling after one wash."
Category:'''}
]

# Role-conditioned: a system prompt sets the persona + strict output rule; the user msg is just the input.
role_conditioned = [
    {'role': 'system', 'content': 'You are a customer-support triage assistant. You always reply with exactly one word from: shipping, quality, sizing, billing.'},
    {'role': 'user', 'content': 'The hem is unravelling after one wash.'}
]

# Run the SAME task through each prompt style and compare the outputs.
print('=== Zero-shot ===')
print(generate(zero_shot, max_new_tokens=20))
print('\n=== Few-shot (3 examples) ===')
print(generate(few_shot, max_new_tokens=20))
print('\n=== Role-conditioned ===')
print(generate(role_conditioned, max_new_tokens=20))

=== Zero-shot ===
Billing

=== Few-shot (3 examples) ===
Complaint: "The box arrived crushed and the package was open."
Category: shipping

=== Role-conditioned ===
Billing.


**Three patterns to know:**

- **Zero-shot** — works for tasks the model has clearly seen during training (sentiment, simple classification). Cheap and quick.
- **Few-shot** — show 2-5 examples. The model "learns" the format from the examples. Powerful for unusual formats, sparse classes, or specific phrasing.
- **Role-conditioned** — system prompt sets behaviour. Best when you want concise output, strict format, or a specific persona.

In production you often combine: role-conditioned for the persona + 2-3 few-shot examples for the format. The combination is more reliable than either alone.

---

## Experiment 2 — Sampling deep-dive

The same prompt with different sampling parameters produces very different distributions of output. Let's MEASURE how much variation each parameter introduces by sampling the same prompt 5 times.

In [4]:
# One fixed prompt -- we vary ONLY the sampling parameters below.
prompt = [
    {'role': 'system', 'content': 'You are a witty product copywriter.'},
    {'role': 'user',   'content': 'Write ONE short tagline for a cosy autumn jumper.'}
]

# Each config is (label, generate kwargs). These map to how the next token is picked:
configs = [
    ('Greedy (T=0)', dict(do_sample=False)),                              # always take the most likely token -> deterministic
    ('Low T=0.3',    dict(do_sample=True, temperature=0.3)),             # low temp = sharpen distribution, little variation
    ('Med T=0.7',    dict(do_sample=True, temperature=0.7)),             # moderate randomness, usually the sweet spot
    ('High T=1.5',   dict(do_sample=True, temperature=1.5)),             # high temp = flatten distribution, wild/creative
    ('Top-p=0.9, T=0.7', dict(do_sample=True, temperature=0.7, top_p=0.9)),  # nucleus sampling: only the top 90% prob mass is eligible
]

for name, cfg in configs:
    print(f"\n=== {name} ===")
    # Loop over seeds: with sampling ON, each seed gives a different draw, so this
    # exposes how much run-to-run VARIATION a given setting produces (greedy stays identical).
    for seed in range(3):
        torch.manual_seed(seed)  # reset RNG so each seed is reproducible
        out = generate(prompt, max_new_tokens=30, **cfg)
        print(f"  seed {seed}: {out[:100]}")  # truncate to 100 chars for tidy output


=== Greedy (T=0) ===
  seed 0: "Stay warm, stay cozy, and stay warm."
  seed 1: "Stay warm, stay cozy, and stay warm."
  seed 2: "Stay warm, stay cozy, and stay warm."

=== Low T=0.3 ===
  seed 0: "Stay warm, stay cozy, and stay snuggly in our cosy autumn jumper."
  seed 1: "Stay warm, stay cozy, and stay with me in the crisp autumn air."
  seed 2: "Stay warm, stay cozy, and stay snuggled up in the chill of autumn."

=== Med T=0.7 ===
  seed 0: "Say goodbye to chilly mornings, hello to warm evenings, and say hello to cozy warmth, this jumper i
  seed 1: "Stay warm, feel comfortable, and enjoy a cozy autumn day."
  seed 2: "Tucked away in the crisp autumn air, a cozy spring jumper will keep you snuggled and warm, ready fo

=== High T=1.5 ===
  seed 0: Jumper To Stay, Dress So High!
  seed 1: "Stay fit for 'Mornin' season—on! Come chill in 'Camel Jerseys or 'Morny,' your cozy
  seed 2: As the chill of autumn coats you in comfort, "A Warm Summer's Bounce" becomes a summer that feels li



Observations:

- **Greedy is identical across seeds** (no sampling = deterministic)
- **Low temperature (0.3)** is nearly identical across seeds; very repetitive
- **Medium temperature (0.7)** gives interesting variation while staying coherent — sweet spot for most use
- **High temperature (1.5)** introduces noticeable drift; might start to break sentence structure
- **Top-p=0.9** narrows the distribution dynamically — protects against the worst high-T failures

**Production rule of thumb:** start with T=0.7 + top-p=0.9. Lower temperature for code/structured output (T=0.1-0.3). Higher temperature for creative writing (T=1.0+).

---

## Experiment 3 — When to use what: prompt vs RAG vs fine-tune

Three options for adapting an LLM to YOUR task. Each has different costs and best-fit scenarios.

In [5]:
# Prints a static ASCII decision tree (prompt -> RAG -> fine-tune); purely illustrative.
print('''
┌──────────────────────────────────────────────────────────────────────┐
│  Decision tree: prompt → RAG → fine-tune                             │
├──────────────────────────────────────────────────────────────────────┤
│                                                                      │
│  START                                                               │
│    │                                                                 │
│    ▼                                                                 │
│  Does the task require KNOWLEDGE the LLM doesn\'t have?               │
│    │                                                                 │
│    ├─ NO  ──► Just PROMPT. Maybe few-shot. Done.                    │
│    │         (sentiment, summarisation, translation,                 │
│    │          format conversion, code completion)                    │
│    │                                                                 │
│    └─ YES ──► Is your knowledge a corpus you can search?             │
│                │                                                     │
│                ├─ YES ──► Use RAG.                                   │
│                │           (customer docs, product catalogue,        │
│                │            internal wiki, code search)              │
│                │                                                     │
│                └─ NO  ──► Does the task need a STYLE                 │
│                           or BEHAVIOUR the model doesn\'t have?       │
│                            │                                         │
│                            ├─ YES ──► FINE-TUNE.                     │
│                            │           (medical reports, legal       │
│                            │            briefs, your brand voice)    │
│                            │                                         │
│                            └─ NO  ──► PROMPT harder. Add examples,   │
│                                       refine system prompt.          │
│                                                                      │
└──────────────────────────────────────────────────────────────────────┘
''')


┌──────────────────────────────────────────────────────────────────────┐
│  Decision tree: prompt → RAG → fine-tune                             │
├──────────────────────────────────────────────────────────────────────┤
│                                                                      │
│  START                                                               │
│    │                                                                 │
│    ▼                                                                 │
│  Does the task require KNOWLEDGE the LLM doesn't have?               │
│    │                                                                 │
│    ├─ NO  ──► Just PROMPT. Maybe few-shot. Done.                    │
│    │         (sentiment, summarisation, translation,                 │
│    │          format conversion, code completion)                    │
│    │                                                                 │
│    └─ YES ──► Is your knowledge a corpus you can 

**Cost order, cheap to expensive:**

1. **Prompt engineering** — free, immediate, no infrastructure
2. **Few-shot prompting** — same cost, slightly higher token budget
3. **RAG** — small infra (embeddings + vector store); modest latency hit
4. **Fine-tuning** — significant compute cost ($100-$10K depending on model size and data) + ongoing maintenance
5. **Pretraining from scratch** — almost never the right choice for a product team

**Cost order, in practice:**

For 90% of business GenAI use cases, the answer is *prompt or RAG, not fine-tune*. Fine-tuning is appropriate when:
- You have stable, repeated tasks (not exploratory) — fine-tuning has high setup cost, amortise it
- You need a specific BEHAVIOUR not knowledge (medical-style writing, brand voice, structured output)
- Prompt-only solutions consistently fail evaluation

If you're not sure, **start with prompt + RAG**. You can always fine-tune later. You can't easily unfine-tune a model.

### A specific example

Sarah's catalogue auto-tagger from L08 — would prompting an LLM work?

In [6]:
# Try classifying a product description by category, prompt-only (no training/fine-tune).
zero_shot_tagger = [
    {'role': 'system', 'content': 'You are a product-category classifier. Reply with exactly one word from: dress, shirt, trouser, coat, knit, footwear, accessory, activewear, home, kids.'},
    {'role': 'user', 'content': 'Lightweight floral frock perfect for warm summer days. Tea-length, pastel print, breathable cotton.'}
]

print('Prompt-only product classifier:')
# do_sample=False -> greedy, so the classification is deterministic.
print(generate(zero_shot_tagger, max_new_tokens=15, do_sample=False))

# A few trickier inputs (incl. a non-clothing distractor) to probe where prompt-only breaks down.
hard_cases = [
    'Soft pique cotton polo in deep indigo. Smart-casual all-rounder.',
    'Wood-fired pizza with fresh mozzarella, basil leaves, and San Marzano tomato sauce.',
    'Stone-washed pure linen duvet set in oat. King-size.',
    'Children\'s rubber wellington boots in navy.',
]
print('\nHarder cases:')
for case in hard_cases:
    # Reuse the same system prompt; feed each case (capped at 90 chars) as the user message.
    msg = [
        {'role': 'system', 'content': zero_shot_tagger[0]['content']},
        {'role': 'user', 'content': case[:90]}
    ]
    cat = generate(msg, max_new_tokens=10, do_sample=False)  # greedy for stable, comparable answers
    print(f"  {case[:60]:60s} -> {cat[:30]}")  # aligned columns: input -> predicted category

Prompt-only product classifier:
Tea-length, pastel print, breathable cotton.

Harder cases:
  Soft pique cotton polo in deep indigo. Smart-casual all-roun -> Smart-casual all-rounder.
  Wood-fired pizza with fresh mozzarella, basil leaves, and Sa -> Wood-fired pizza with fresh mo
  Stone-washed pure linen duvet set in oat. King-size.         -> Knit.
  Children's rubber wellington boots in navy.                  -> Navy.


**The result is decent but imperfect.** A small (360M) LLM with zero-shot prompting gets simple cases right but slips on edge cases. A larger model (Claude, GPT-4) would do better. A fine-tuned classifier (L08's CNN) does even better because it was specifically optimised for this task.

**The lesson:** for narrow, repeatable classification tasks, fine-tuning a small model often beats prompting a large one. For open-ended language tasks, prompting wins. Know the task before picking the tool.

---

## Recap

| Technique | Use when |
|-----------|----------|
| Zero-shot prompt | Task is common, model has seen it during training |
| Few-shot prompt  | Format is specific or class is rare; show 2-5 examples |
| Role-conditioned | You want a persona, strict format, or constrained output |
| Low temperature  | Code, structured output, factual responses |
| Higher temperature | Creative writing, brainstorming |
| RAG              | Need YOUR data; the corpus is searchable |
| Fine-tuning      | Stable repeated task that prompting can't solve reliably |

You've now seen the full GenAI production toolkit. The mathematics took 50 years to invent; the production patterns took five. Picking the right one for YOUR task is the new differentiator.